<a href="https://colab.research.google.com/github/tijesu24/Plant-disease-Recognition-papers/blob/main/Plantseg%20wt%20starnet%20replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
weitianqi_plantseg_path = kagglehub.dataset_download('weitianqi/plantseg')

print('Data source import complete.')


Data source import complete.


# Cell 1: Environment Setup & Dependencies

In [14]:
!pip install -q timm albumentations segment-anything

import os
import re
import time
import math
import urllib.request
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from segment_anything import sam_model_registry

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Download official Meta SAM ViT-B weights (SA-1B pretrained, as specified in Section 3.3)
SAM_CKPT_PATH = "/kaggle/working/sam_vit_b_01ec64.pth"
if not os.path.exists(SAM_CKPT_PATH):
    print("Downloading SAM ViT-B checkpoint...")
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
        SAM_CKPT_PATH
    )
    print("SAM ViT-B checkpoint downloaded.")

Using device: cuda


# Cell 2: Dataset & Augmentations (Reusing Your PlantSeg Paths)

In [15]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}
IGNORE_INDEX = 255

# Paper Section 3.1: 512x512 input, horizontal flip, vertical flip, random rotation
train_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=30, border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=IGNORE_INDEX, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])


def clean_label(stem):
    label = re.sub(r"_(google|bing|baidu)", "", stem, flags=re.IGNORECASE)
    label = re.sub(r"_([a-zA-Z]+_[a-zA-Z]+)\s*\(\d+\)", "", label)
    label = re.sub(r"_([a-zA-Z]+)\s*\(\d+\)", "", label)
    label = re.sub(r"\s*\(\d+\)", "", label)
    label = re.sub(r"_\d+$", "", label)
    return label.lower().strip()


class PlantSegReLeafDataset(Dataset):
    """
    Reuses your PlantSeg directory layout and converts multi-class lesion masks
    into binary lesion masks (0 = background, 1 = disease lesion) per ReLeaf-SAM.
    """
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = Path(img_dir)
        self.mask_dir = Path(mask_dir)
        self.transform = transform

        img_files = {f.stem: f for f in self.img_dir.glob("*.*") if f.suffix.lower() in IMAGE_SUFFIXES}
        mask_files = {f.stem: f for f in self.mask_dir.glob("*.*") if f.suffix.lower() in IMAGE_SUFFIXES}
        common_stems = sorted(set(img_files) & set(mask_files))
        self.samples = [(img_files[stem], mask_files[stem]) for stem in common_stems]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        image = cv2.imread(str(img_path))
        if image is None:
            raise ValueError(f"Could not read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        raw_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if raw_mask is None:
            raise ValueError(f"Could not read mask: {mask_path}")

        # Convert any lesion class ID (> 0) into foreground class 1
        binary_mask = (raw_mask > 0).astype(np.uint8)

        if self.transform:
            augmented = self.transform(image=image, mask=binary_mask)
            image = augmented["image"]
            mask = augmented["mask"].long()
        else:
            mask = torch.tensor(binary_mask, dtype=torch.long)

        return image, mask

# Cell 3: ReLeaf-SAM Architecture (Detail Compensation + Uncertainty Encoder + Variational Fusion + SAM Decoder)

In [16]:
class BranchMLP(nn.Module):
    """
    Two-layer MLP described in Section 2.2.2 & 2.2.3:
    Linear(in_c, in_c // 2) -> BN -> ReLU -> Linear(in_c // 2, out_c)
    Implemented with 1x1 Conv2d + BatchNorm2d for spatial [B, C, H, W] feature maps.
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        mid_channels = in_channels // 2
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=1, bias=True)
        )

    def forward(self, x):
        return self.net(x)


class UncertaintyEncoder(nn.Module):
    """
    Section 2.2.2 (Equations 1-6):
    Runs in float32 to prevent fp16 overflow in exp(0.5 * x) on Kaggle T4 GPUs.
    """
    def __init__(self, sam_in_channels=256, cnn_in_channels=2048, latent_dim=768):
        super().__init__()
        self.proj_c = nn.Sequential(
            nn.ConvTranspose2d(cnn_in_channels, latent_dim, kernel_size=2, stride=2, bias=False),
            nn.BatchNorm2d(latent_dim),
            nn.ReLU(inplace=True)
        )
        self.proj_s = nn.Sequential(
            nn.Conv2d(sam_in_channels, latent_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(latent_dim),
            nn.ReLU(inplace=True)
        )

        self.mlp_mu_c = BranchMLP(latent_dim, latent_dim)
        self.mlp_std_c = BranchMLP(latent_dim, latent_dim)
        self.mlp_mu_s = BranchMLP(latent_dim, latent_dim)
        self.mlp_std_s = BranchMLP(latent_dim, latent_dim)

    def forward(self, f_s, f_c):
        with torch.amp.autocast("cuda", enabled=False):
            f_s = f_s.float()
            f_c = f_c.float()

            z_s = self.proj_s(f_s)  # [B, 768, 32, 32]
            z_c = self.proj_c(f_c)  # [B, 768, 32, 32]

            mu_s = torch.clamp(self.mlp_mu_s(z_s), min=-10.0, max=10.0)
            raw_std_s = torch.clamp(self.mlp_std_s(z_s), min=-6.0, max=2.0)
            sigma_s = torch.exp(0.5 * raw_std_s)

            mu_c = torch.clamp(self.mlp_mu_c(z_c), min=-10.0, max=10.0)
            raw_std_c = torch.clamp(self.mlp_std_c(z_c), min=-6.0, max=2.0)
            sigma_c = torch.exp(0.5 * raw_std_c)

            if self.training:
                e_s = mu_s + sigma_s * torch.randn_like(sigma_s)
                e_c = mu_c + sigma_c * torch.randn_like(sigma_c)
            else:
                e_s = mu_s
                e_c = mu_c

        return z_s, z_c, e_s, e_c, (mu_s, sigma_s, mu_c, sigma_c)


class VariationalFusion(nn.Module):
    """
    Section 2.2.3 (Equations 7-9):
    Runs in float32 to prevent fp16 overflow in attention sampling and softmax.
    """
    def __init__(self, latent_dim=768, out_channels=256):
        super().__init__()
        self.mlp_mu_k_s = BranchMLP(latent_dim, 1)
        self.mlp_std_k_s = BranchMLP(latent_dim, 1)
        self.mlp_mu_k_c = BranchMLP(latent_dim, 1)
        self.mlp_std_k_c = BranchMLP(latent_dim, 1)

        self.w_l = nn.Parameter(torch.ones(1))

        self.out_proj = nn.Sequential(
            nn.Conv2d(latent_dim, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )

    def forward(self, z_s, z_c, e_s, e_c):
        with torch.amp.autocast("cuda", enabled=False):
            z_s, z_c, e_s, e_c = z_s.float(), z_c.float(), e_s.float(), e_c.float()

            mu_k_s = torch.clamp(self.mlp_mu_k_s(e_s), min=-10.0, max=10.0)
            raw_std_k_s = torch.clamp(self.mlp_std_k_s(e_s), min=-6.0, max=2.0)
            sigma_k_s = torch.exp(0.5 * raw_std_k_s)

            mu_k_c = torch.clamp(self.mlp_mu_k_c(e_c), min=-10.0, max=10.0)
            raw_std_k_c = torch.clamp(self.mlp_std_k_c(e_c), min=-6.0, max=2.0)
            sigma_k_c = torch.exp(0.5 * raw_std_k_c)

            if self.training:
                a_hat_s = mu_k_s + sigma_k_s * torch.randn_like(sigma_k_s)
                a_hat_c = mu_k_c + sigma_k_c * torch.randn_like(sigma_k_c)
            else:
                a_hat_s = mu_k_s
                a_hat_c = mu_k_c

            logits = torch.cat([a_hat_s, a_hat_c], dim=1)  # [B, 2, 32, 32]
            weights = F.softmax(logits, dim=1)
            w_s = weights[:, 0:1, :, :]
            w_c = weights[:, 1:2, :, :]

            f_fusion = self.w_l * (w_s * z_s + w_c * z_c)  # [B, 768, 32, 32]
            f_decoder_in = self.out_proj(f_fusion)         # [B, 256, 32, 32]

        return f_decoder_in, (mu_k_s, sigma_k_s, mu_k_c, sigma_k_c)


class ReLeafSAM(nn.Module):
    """
    Full ReLeaf-SAM Architecture (Liu et al., 2026).
    """
    def __init__(self, sam_checkpoint, img_size=512, num_classes=2):
        super().__init__()
        self.img_size = img_size

        # 1. Global Modeling Branch: Pre-trained SAM ViT-B
        sam = sam_model_registry["vit_b"](checkpoint=sam_checkpoint)
        self._adapt_sam_pos_embed(sam, img_size=img_size)
        self.sam_image_encoder = sam.image_encoder
        self.sam_prompt_encoder = sam.prompt_encoder
        self.sam_mask_decoder = sam.mask_decoder

        for param in self.sam_prompt_encoder.parameters():
            param.requires_grad = False

        # 2. Detail Compensation Branch: ResNet-50 (ImageNet pretrained)
        self.detail_branch = timm.create_model(
            "resnet50",
            pretrained=True,
            features_only=True,
            out_indices=(4,)
        )

        # 3. Uncertainty Encoder & Variational Fusion
        self.uncertainty_encoder = UncertaintyEncoder(
            sam_in_channels=256,
            cnn_in_channels=2048,
            latent_dim=768
        )
        self.variational_fusion = VariationalFusion(
            latent_dim=768,
            out_channels=256
        )

        # 4. Binary segmentation head initialized cleanly to preserve SAM logit polarity
        self.cls_head = nn.Conv2d(1, num_classes, kernel_size=1)
        with torch.no_grad():
            self.cls_head.weight.copy_(torch.tensor([[[-0.5]], [[0.5]]]).view(2, 1, 1, 1))
            self.cls_head.bias.zero_()

    @staticmethod
    def _adapt_sam_pos_embed(sam, img_size=512):
        """Interpolates SAM's 64x64 abs pos_embed and 127-len rel_pos embeddings to 32x32 / 63."""
        old_pos = sam.image_encoder.pos_embed  # [1, 64, 64, 768]
        new_grid = img_size // 16              # 32 for 512x512
        if old_pos.shape[1] != new_grid:
            pos_permuted = old_pos.permute(0, 3, 1, 2)
            new_pos = F.interpolate(pos_permuted, size=(new_grid, new_grid), mode="bicubic", align_corners=False)
            sam.image_encoder.pos_embed = nn.Parameter(new_pos.permute(0, 2, 3, 1))
            sam.image_encoder.img_size = img_size

            new_rel_len = 2 * new_grid - 1  # 63 for 32x32
            for blk in sam.image_encoder.blocks:
                if hasattr(blk.attn, "rel_pos_h") and blk.attn.rel_pos_h.shape[0] != new_rel_len:
                    rel_h = blk.attn.rel_pos_h.permute(1, 0).unsqueeze(0)
                    rel_h = F.interpolate(rel_h, size=new_rel_len, mode="linear", align_corners=False)
                    blk.attn.rel_pos_h = nn.Parameter(rel_h.squeeze(0).permute(1, 0))

                    rel_w = blk.attn.rel_pos_w.permute(1, 0).unsqueeze(0)
                    rel_w = F.interpolate(rel_w, size=new_rel_len, mode="linear", align_corners=False)
                    blk.attn.rel_pos_w = nn.Parameter(rel_w.squeeze(0).permute(1, 0))

    def forward(self, x):
        B, _, H, W = x.shape

        f_s = self.sam_image_encoder(x)        # [B, 256, 32, 32]
        f_c = self.detail_branch(x)[0]         # [B, 2048, 16, 16]

        z_s, z_c, e_s, e_c, ue_params = self.uncertainty_encoder(f_s, f_c)
        f_fused, vf_params = self.variational_fusion(z_s, z_c, e_s, e_c)  # [B, 256, 32, 32]

        sparse_embeddings, dense_embeddings = self.sam_prompt_encoder(
            points=None, boxes=None, masks=None
        )
        if dense_embeddings.shape[-2:] != f_fused.shape[-2:]:
            dense_embeddings = F.interpolate(
                dense_embeddings, size=f_fused.shape[-2:], mode="bilinear", align_corners=False
            )

        image_pe = self.sam_prompt_encoder.get_dense_pe()
        if image_pe.shape[-2:] != f_fused.shape[-2:]:
            image_pe = F.interpolate(
                image_pe, size=f_fused.shape[-2:], mode="bilinear", align_corners=False
            )

        low_res_masks, _ = self.sam_mask_decoder(
            image_embeddings=f_fused,
            image_pe=image_pe,
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=False
        )  # [B, 1, 128, 128]

        upsampled_mask = F.interpolate(low_res_masks, size=(H, W), mode="bilinear", align_corners=False)
        logits = self.cls_head(upsampled_mask)  # [B, 2, 512, 512]

        return logits, ue_params, vf_params

# Cell 4: Loss Function ($L = L_{KL} + L_{Dice} + L_{CE}$, Equations 10–12)

In [17]:
class ReLeafSAMLoss(nn.Module):
    """
    Section 2.3 (Equations 10-12):
    L = beta_kl * L_KL + L_Dice + L_CE
    """
    def __init__(self, ignore_index=255, smooth=1e-5, kl_weight=0.01):
        super().__init__()
        self.ignore_index = ignore_index
        self.smooth = smooth
        self.kl_weight = kl_weight
        self.ce = nn.CrossEntropyLoss(ignore_index=ignore_index)

    @staticmethod
    def kl_standard_normal(mu, sigma):
        mu = mu.float()
        sigma = sigma.float()
        var = sigma.pow(2)
        kl = 0.5 * (var + mu.pow(2) - torch.log(var + 1e-8) - 1.0)
        return kl.mean()

    def dice_loss(self, logits, targets):
        probs = F.softmax(logits.float(), dim=1)[:, 1]
        valid_mask = (targets != self.ignore_index)
        gt = (targets == 1).float() * valid_mask.float()
        probs = probs * valid_mask.float()

        intersection = (probs * gt).sum(dim=(1, 2))
        union = probs.sum(dim=(1, 2)) + gt.sum(dim=(1, 2))
        dsc = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dsc.mean()

    def forward(self, logits, targets, ue_params, vf_params):
        mu_s, sigma_s, mu_c, sigma_c = ue_params
        mu_k_s, sigma_k_s, mu_k_c, sigma_k_c = vf_params

        l_kl_raw = (
            self.kl_standard_normal(mu_s, sigma_s)
            + self.kl_standard_normal(mu_c, sigma_c)
            + self.kl_standard_normal(mu_k_s, sigma_k_s)
            + self.kl_standard_normal(mu_k_c, sigma_k_c)
        )
        l_kl = self.kl_weight * l_kl_raw

        l_dice = self.dice_loss(logits, targets)
        l_ce = self.ce(logits.float(), targets)

        total_loss = l_kl + l_dice + l_ce
        return total_loss, (l_ce.item(), l_dice.item(), l_kl.item())

# Cell 5: Kaggle-Optimized Training, Paper Metrics (DSC, IoU, Precision, Recall), Periodic Checkpointing & Visualization

In [18]:
import gc

def compute_binary_metrics(preds, targets, ignore_index=255):
    """Computes Equations 13-16: DSC, IoU, Precision, Recall."""
    valid = (targets != ignore_index)
    pred_fg = (preds == 1) & valid
    gt_fg = (targets == 1) & valid

    tp = (pred_fg & gt_fg).sum().item()
    fp = (pred_fg & (~gt_fg)).sum().item()
    fn = ((~pred_fg) & gt_fg).sum().item()
    return tp, fp, fn


def visualize_releaf_predictions(model, val_dataset, device, num_samples=4):
    """Visualizes Input, Ground Truth (blue overlay), Prediction (blue overlay), and Error Map."""
    print("\nGenerating ReLeaf-SAM prediction visualizations...")
    eval_model = model.module if isinstance(model, nn.DataParallel) else model
    eval_model.eval()

    rng = np.random.default_rng(42)
    indices = rng.choice(len(val_dataset), size=min(num_samples, len(val_dataset)), replace=False)

    fig, axes = plt.subplots(len(indices), 4, figsize=(16, 4 * len(indices)))
    if len(indices) == 1:
        axes = np.expand_dims(axes, axis=0)

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    with torch.no_grad():
        for row_idx, sample_idx in enumerate(indices):
            img_tensor, gt_mask = val_dataset[sample_idx]
            input_batch = img_tensor.unsqueeze(0).to(device)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                logits, _, _ = eval_model(input_batch)

            pred_mask = logits.argmax(dim=1).squeeze(0).cpu().numpy()
            gt_np = gt_mask.cpu().numpy()

            img_np = np.clip(img_tensor.permute(1, 2, 0).cpu().numpy() * std + mean, 0, 1)

            gt_overlay = img_np.copy()
            gt_overlay[gt_np == 1] = gt_overlay[gt_np == 1] * 0.45 + np.array([0.1, 0.1, 0.95]) * 0.55

            pred_overlay = img_np.copy()
            pred_overlay[pred_mask == 1] = pred_overlay[pred_mask == 1] * 0.45 + np.array([0.1, 0.1, 0.95]) * 0.55

            tp = (gt_np == 1) & (pred_mask == 1)
            fn = (gt_np == 1) & (pred_mask == 0)
            fp = (gt_np == 0) & (pred_mask == 1)
            err_map = np.zeros((*gt_np.shape, 3), dtype=np.uint8)
            err_map[tp] = [255, 255, 255]
            err_map[fn] = [255, 0, 0]
            err_map[fp] = [0, 100, 255]

            dsc = (2 * tp.sum()) / (2 * tp.sum() + fp.sum() + fn.sum() + 1e-7) * 100.0

            axes[row_idx, 0].imshow(img_np)
            axes[row_idx, 0].set_title(f"Val #{sample_idx}: Raw Image")
            axes[row_idx, 0].axis("off")

            axes[row_idx, 1].imshow(gt_overlay)
            axes[row_idx, 1].set_title("Ground Truth (Blue)")
            axes[row_idx, 1].axis("off")

            axes[row_idx, 2].imshow(pred_overlay)
            axes[row_idx, 2].set_title(f"ReLeaf-SAM (DSC: {dsc:.1f}%)")
            axes[row_idx, 2].axis("off")

            axes[row_idx, 3].imshow(err_map)
            axes[row_idx, 3].set_title("Error Map (White=TP, Red=FN, Blue=FP)")
            axes[row_idx, 3].axis("off")

    plt.tight_layout()
    vis_path = "/kaggle/working/releaf_sam_predictions.png"
    plt.savefig(vis_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved visualization to {vis_path}")


def train_releaf_sam(
    resume_checkpoint=None,
    total_epochs=100,          # 100 epochs reaches strong convergence with pretrained SAM + ResNet-50
    max_session_hours=11.0,    # Auto-stops cleanly before Kaggle's 12h timeout
    per_gpu_batch_size=4,      # 4 per GPU (8 on T4 x2)
    effective_batch_size=16,   # Matches paper's batch_size=16 via gradient accumulation
    save_every=5
):
    session_start = time.perf_counter()
    num_gpus = max(1, torch.cuda.device_count())
    batch_size = per_gpu_batch_size * num_gpus
    grad_accum_steps = max(1, effective_batch_size // batch_size)
    print(f"GPUs detected: {num_gpus} | Micro-batch: {batch_size} | Grad accum: {grad_accum_steps} (Effective BS={batch_size * grad_accum_steps})")

    dataset_root = Path("/kaggle/input/datasets/weitianqi/plantseg/plantsegv2")
    train_dataset = PlantSegReLeafDataset(
        img_dir=dataset_root / "images" / "train",
        mask_dir=dataset_root / "annotations" / "train",
        transform=train_transform
    )
    val_dataset = PlantSegReLeafDataset(
        img_dir=dataset_root / "images" / "val",
        mask_dir=dataset_root / "annotations" / "val",
        transform=val_transform
    )
    print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")

    # num_workers=2 with persistent_workers keeps CPU RAM flat while speeding up OpenCV reads
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True, persistent_workers=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size * 2, shuffle=False,
        num_workers=2, pin_memory=True, persistent_workers=True
    )

    base_model = ReLeafSAM(sam_checkpoint=SAM_CKPT_PATH, img_size=512, num_classes=2).to(device)
    gc.collect()

    if num_gpus > 1:
        model = nn.DataParallel(base_model)
    else:
        model = base_model

    criterion = ReLeafSAMLoss(ignore_index=IGNORE_INDEX, kl_weight=0.01).to(device)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=1e-4,
        weight_decay=1e-4
    )

    warmup_epochs = 2
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs - warmup_epochs, eta_min=1e-6)
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs]
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    start_epoch = 0
    best_dsc = 0.0
    ckpt_dir = Path("/kaggle/working/checkpoints")
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    latest_ckpt_path = ckpt_dir / "latest_checkpoint.pth"

    if resume_checkpoint is None and latest_ckpt_path.exists():
        latest_ckpt_path.unlink()

    ckpt_to_load = latest_ckpt_path if latest_ckpt_path.exists() else (Path(resume_checkpoint) if resume_checkpoint else None)
    if ckpt_to_load and ckpt_to_load.exists():
        print(f"Resuming from checkpoint: {ckpt_to_load}")
        ckpt = torch.load(ckpt_to_load, map_location=device)
        raw_model = model.module if isinstance(model, nn.DataParallel) else model
        raw_model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        scaler.load_state_dict(ckpt["scaler_state"])
        start_epoch = ckpt["epoch"] + 1
        best_dsc = ckpt.get("best_dsc", 0.0)
        print(f"Resumed at Epoch {start_epoch} (Best Val DSC so far: {best_dsc:.2f}%)")

    for epoch in range(start_epoch, total_epochs):
        # Stop gracefully if approaching Kaggle's 12h limit
        elapsed_hours = (time.perf_counter() - session_start) / 3600.0
        if elapsed_hours >= max_session_hours:
            print(f"Reached {elapsed_hours:.2f}h session budget. Stopping cleanly so outputs are saved.")
            break

        model.train()
        epoch_start = time.perf_counter()
        running_loss = 0.0
        valid_steps = 0
        optimizer.zero_grad(set_to_none=True)

        for step, (images, masks) in enumerate(train_loader, start=1):
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                logits, ue_params, vf_params = model(images)
                loss, (l_ce, l_dice, l_kl) = criterion(logits, masks, ue_params, vf_params)

            if not torch.isfinite(loss):
                print(f"Warning: Non-finite loss at Epoch {epoch + 1}, Step {step}. Skipping batch.")
                optimizer.zero_grad(set_to_none=True)
                continue

            scaled_loss = loss / grad_accum_steps
            scaler.scale(scaled_loss).backward()

            if step % grad_accum_steps == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            running_loss += loss.item()
            valid_steps += 1

            if step % 150 == 0:
                print(
                    f"Epoch [{epoch + 1}/{total_epochs}] | Step [{step}/{len(train_loader)}] | "
                    f"Loss: {loss.item():.4f} (CE: {l_ce:.3f}, Dice: {l_dice:.3f}, KL: {l_kl:.4f}) | "
                    f"LR: {optimizer.param_groups[0]['lr']:.2e}"
                )

        scheduler.step()

        # Validation Loop
        model.eval()
        total_tp, total_fp, total_fn = 0, 0, 0
        with torch.no_grad():
            for val_images, val_masks in val_loader:
                val_images = val_images.to(device, non_blocking=True)
                val_masks = val_masks.to(device, non_blocking=True)

                with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                    val_logits, _, _ = model(val_images)

                preds = val_logits.argmax(dim=1)
                tp, fp, fn = compute_binary_metrics(preds, val_masks, ignore_index=IGNORE_INDEX)
                total_tp += tp
                total_fp += fp
                total_fn += fn

        dsc = (2.0 * total_tp) / (2.0 * total_tp + total_fp + total_fn + 1e-7) * 100.0
        iou = total_tp / (total_tp + total_fp + total_fn + 1e-7) * 100.0
        precision = total_tp / (total_tp + total_fp + 1e-7) * 100.0
        recall = total_tp / (total_tp + total_fn + 1e-7) * 100.0

        elapsed_min = (time.perf_counter() - epoch_start) / 60.0
        avg_train_loss = running_loss / max(valid_steps, 1)
        print(f"\n--- Epoch [{epoch + 1}/{total_epochs}] Summary ({elapsed_min:.1f} min) ---")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val DSC: {dsc:.2f}% | Val IoU: {iou:.2f}% | Precision: {precision:.2f}% | Recall: {recall:.2f}%\n")

        if math.isfinite(avg_train_loss) and valid_steps > 0:
            raw_model = model.module if isinstance(model, nn.DataParallel) else model
            state_dict = {
                "epoch": epoch,
                "model_state": raw_model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "scaler_state": scaler.state_dict(),
                "best_dsc": max(best_dsc, dsc)
            }
            torch.save(state_dict, latest_ckpt_path)

            if dsc > best_dsc:
                best_dsc = dsc
                torch.save(state_dict, ckpt_dir / "best_releaf_sam.pth")
                print(f"New best model saved! (Val DSC: {best_dsc:.2f}%)")

            if (epoch + 1) % save_every == 0:
                periodic_path = ckpt_dir / f"releaf_sam_epoch_{epoch + 1}.pth"
                torch.save(state_dict, periodic_path)
                print(f"Periodic checkpoint saved: {periodic_path}")
        else:
            print("Warning: Skipping checkpoint save due to non-finite loss.")

    visualize_releaf_predictions(model, val_dataset, device, num_samples=4)
    print("Session complete.")


train_releaf_sam(
    resume_checkpoint=None,
    total_epochs=100,
    max_session_hours=11.0,
    per_gpu_batch_size=4,
    effective_batch_size=16,
    save_every=5
)

GPUs detected: 2 | Micro-batch: 8 | Grad accum: 2 (Effective BS=16)
Train samples: 7916 | Val samples: 1247
Epoch [1/100] | Step [150/990] | Loss: 1.3525 (CE: 0.639, Dice: 0.709, KL: 0.0049) | LR: 1.00e-05
Epoch [1/100] | Step [300/990] | Loss: 0.8934 (CE: 0.349, Dice: 0.539, KL: 0.0051) | LR: 1.00e-05
Epoch [1/100] | Step [450/990] | Loss: 0.7397 (CE: 0.291, Dice: 0.443, KL: 0.0060) | LR: 1.00e-05
Epoch [1/100] | Step [600/990] | Loss: 0.6739 (CE: 0.264, Dice: 0.403, KL: 0.0068) | LR: 1.00e-05


KeyboardInterrupt: 